# Exploratory Data Analysis
Global Cybersecurity Threats (2015-2024) — Fast vs. Slow Resolution project

First working version, written without the Colab/reference material yet.
Will be revised with attribution comments once those are provided.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

RAW_PATH = Path("../data/raw/cybersecurity_threats.csv")
FIGURES_DIR = Path("../outputs/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")

In [ ]:
df = pd.read_csv(RAW_PATH)
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(r"[^\w\s]", "", regex=True)
    .str.replace(r"\s+", "_", regex=True)
)
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

## 2. Class balance check — will Fast/Slow be roughly balanced?

In [ ]:
median_time = df["incident_resolution_time_in_hours"].median()
print(f"Median resolution time: {median_time:.2f} hours")

df["resolution_class"] = (
    df["incident_resolution_time_in_hours"] >= median_time
).map({True: "Slow", False: "Fast"})

df["resolution_class"].value_counts()

In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(data=df, x="resolution_class")
plt.title("Fast vs. Slow Resolution Class Balance")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "class_balance.png")
plt.show()

## 3. Average resolution time by defense mechanism
This is the core question for Option C — which defense mechanisms are
associated with faster containment?

In [ ]:
avg_by_defense = (
    df.groupby("defense_mechanism_used")["incident_resolution_time_in_hours"]
    .mean()
    .sort_values()
)
avg_by_defense

In [ ]:
plt.figure(figsize=(7, 5))
sns.barplot(x=avg_by_defense.values, y=avg_by_defense.index, orient="h")
plt.xlabel("Average Resolution Time (hours)")
plt.title("Average Resolution Time by Defense Mechanism")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "avg_resolution_by_defense.png")
plt.show()

## 4. Fast/Slow ratio by defense mechanism

In [ ]:
ratio_table = pd.crosstab(
    df["defense_mechanism_used"], df["resolution_class"], normalize="index"
)
ratio_table

In [ ]:
ratio_table.plot(kind="barh", stacked=True, figsize=(7, 5), colormap="coolwarm")
plt.xlabel("Proportion")
plt.title("Fast vs. Slow Proportion by Defense Mechanism")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "fast_slow_ratio_by_defense.png")
plt.show()

## 5. Defense mechanism vs. attack type
Does the 'best' defense mechanism depend on the type of attack?

In [ ]:
crosstab = pd.crosstab(df["attack_type"], df["defense_mechanism_used"])

plt.figure(figsize=(9, 6))
sns.heatmap(crosstab, annot=True, fmt="d", cmap="YlGnBu")
plt.title("Attack Type vs. Defense Mechanism (counts)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "attack_type_vs_defense.png")
plt.show()

## 6. Notes / takeaways
_(fill in after running — e.g. which defense mechanism had the lowest
average resolution time, whether the effect holds across attack types,
anything surprising in the class balance)_

## 7. Model Training Diagnostics — Loss & Accuracy Curves

**Important context before these plots:** Logistic Regression and Random
Forest don't train in "epochs" the way a neural network does, so there's
no loss/accuracy-per-epoch curve to draw for them. XGBoost, however, is
trained in a series of **boosting rounds** (one new tree added per round),
and its round-by-round training/validation error IS directly analogous to
an epoch curve — that's what the plots below show.

**Sources for this section:**
- Learning curve concept + evals_result() usage:
  https://machinelearningmastery.com/tune-xgboost-performance-with-learning-curves/
- eval_set / eval_metric=["error","logloss"] pattern, plotting train vs. test:
  https://www.projectpro.io/recipes/evaluate-xgboost-model-with-learning-curves-example-2
- Official XGBoost evals_result() API example:
  https://xgboost.readthedocs.io/en/stable/python/examples/sklearn_evals_result.html

This requires `train.py` to have been run at least once (so `models/xgboost.pkl`
exists) — we reuse its tuned hyperparameters rather than guessing new ones.

In [ ]:
import sys
import json
import joblib
from sklearn.model_selection import train_test_split

# Reuse the exact same cleaning/encoding logic as preprocess.py instead of
# duplicating it, so the notebook and the pipeline never drift apart.
sys.path.append(str(Path("../src").resolve()))
import preprocess as pp

PROCESSED_PATH = Path("../data/processed/cyber_processed.csv")
MODELS_DIR = Path("../models")

processed_df = pd.read_csv(PROCESSED_PATH)
X = processed_df.drop(columns=["resolution_class"])
y_raw = processed_df["resolution_class"]

label_encoder = joblib.load(MODELS_DIR / "label_encoder.pkl")
y = label_encoder.transform(y_raw)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train size: {len(X_train)}   Test size: {len(X_test)}")

In [ ]:
from xgboost import XGBClassifier

# Load the ALREADY-TUNED xgboost model from train.py and copy its best
# hyperparameters, so this notebook's curve reflects the real tuned model
# rather than an arbitrary guess at settings.
tuned_xgb = joblib.load(MODELS_DIR / "xgboost.pkl")
best_params = tuned_xgb.get_params()

# Refit with the same hyperparameters, but this time pass eval_set so
# XGBoost records train/validation metrics at every boosting round.
# (The saved model from train.py didn't track this because RandomizedSearchCV
# doesn't expose eval_set through its cross-validation folds.)
diagnostic_model = XGBClassifier(**{**best_params, "eval_metric": ["logloss", "error"]})

eval_set = [(X_train, y_train), (X_test, y_test)]
diagnostic_model.fit(X_train, y_train, eval_set=eval_set, verbose=False)

evals_result = diagnostic_model.evals_result()
n_rounds = len(evals_result["validation_0"]["logloss"])
print(f"Trained for {n_rounds} boosting rounds (~ 'epochs')")

### 7a. Training Loss Curve (Log Loss per Boosting Round)

Log loss measures how confident and correct the model's predicted
probabilities are — lower is better. If the **training** line keeps
dropping while the **validation** line flattens or rises, that's a sign
of overfitting (the model is memorizing the training data rather than
learning patterns that generalize).

In [ ]:
x_axis = range(n_rounds)

plt.figure(figsize=(8, 5))
plt.plot(x_axis, evals_result["validation_0"]["logloss"], label="Train")
plt.plot(x_axis, evals_result["validation_1"]["logloss"], label="Validation (Test)")
plt.xlabel("Boosting Round (~Epoch)")
plt.ylabel("Log Loss")
plt.title("XGBoost Training vs. Validation Log Loss")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "xgboost_loss_curve.png")
plt.show()

### 7b. Epoch Accuracy Curve (per Boosting Round)

XGBoost's `error` metric is the misclassification rate, so accuracy is
simply `1 - error`. This shows how quickly the model's accuracy improves
as more trees are added, and whether it plateaus (or gets worse on the
validation set) after a certain number of rounds.

In [ ]:
train_accuracy = [1 - e for e in evals_result["validation_0"]["error"]]
test_accuracy = [1 - e for e in evals_result["validation_1"]["error"]]

plt.figure(figsize=(8, 5))
plt.plot(x_axis, train_accuracy, label="Train")
plt.plot(x_axis, test_accuracy, label="Validation (Test)")
plt.xlabel("Boosting Round (~Epoch)")
plt.ylabel("Accuracy")
plt.title("XGBoost Training vs. Validation Accuracy")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "xgboost_accuracy_curve.png")
plt.show()

print(f"Final train accuracy: {train_accuracy[-1]:.3f}")
print(f"Final validation accuracy: {test_accuracy[-1]:.3f}")

### 7c. General Accuracy Comparison Across All Three Models

This loads the actual saved models from `train.py` (Logistic Regression,
Random Forest, XGBoost) and compares their accuracy on the same held-out
test set side by side — a general accuracy overview rather than a
per-round curve.

In [ ]:
from sklearn.metrics import accuracy_score

TEST_SPLIT_PATH = Path("../data/processed/test_split.csv")
test_df = pd.read_csv(TEST_SPLIT_PATH)
X_test_saved = test_df.drop(columns=["resolution_class"])
y_test_saved = test_df["resolution_class"]

model_files = {
    "Logistic Regression": "logistic_regression.pkl",
    "Random Forest": "random_forest.pkl",
    "XGBoost": "xgboost.pkl",
}

accuracies = {}
for name, filename in model_files.items():
    path = MODELS_DIR / filename
    if path.exists():
        model = joblib.load(path)
        preds = model.predict(X_test_saved)
        accuracies[name] = accuracy_score(y_test_saved, preds)
    else:
        print(f"{filename} not found - run train.py first")

accuracies

In [ ]:
plt.figure(figsize=(6, 4))
sns.barplot(x=list(accuracies.keys()), y=list(accuracies.values()))
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.title("Test Accuracy by Model")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "model_accuracy_comparison.png")
plt.show()

### 7d. Notes / takeaways
_(fill in after running — e.g. whether the validation loss/accuracy
curves show signs of overfitting, and which model came out on top in
the general accuracy comparison)_